# 01 - Dataset understanding and EDA (Member A)

**Partitions used:** `train` (80 trips) and `student_test` (10 trips). Evaluator-test data is never touched.

This notebook writes:
- `results/dataset_inventory.csv` - one row per trip (sizes, counts, gaps, graph checks, distance to route)
- `results/data_quality.csv` - anomaly log (one row per issue found)
- figures in `reports/figures/` and two Folium maps in `reports/maps/`

## 1. Setup

In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import folium, shapely
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
warnings.filterwarnings("ignore")

# find the repo root whether the notebook is run from the repo root or from notebooks/
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pathsnap").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from pathsnap.config import load_yaml, RANDOM_SEED
from pathsnap.data import load_trip, list_trip_dirs

cfg = load_yaml("configs/paths.yaml")
RAW = REPO_ROOT / "data" / "raw"
# accept data/raw/train  OR  data/raw/Student folder/train
BASE = RAW if (RAW / "train").exists() else RAW / "Student folder"
PARTS = {"train": BASE / "train", "student_test": BASE / "student_test"}
RESULTS = REPO_ROOT / cfg["artifacts"]["results_dir"]
FIGS = REPO_ROOT / cfg["artifacts"]["figures_dir"]
MAPS = REPO_ROOT / cfg["artifacts"]["maps_dir"]
for p in (RESULTS, FIGS, MAPS):
    p.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

for name, p in PARTS.items():
    print(f"{name:13s} {p}  ->  {len(list_trip_dirs(p))} trips")

## 2. One pass over every trip

For each trip we compute (all from the raw files, nothing typed by hand):

- file sizes, number of GPS points / nodes / arcs / route arcs
- missing or invalid GPS values, timestamp order, time gaps, share of irregular time steps
- geographical extent and travelled distance (metres, from the per-trip local projection in `load_trip`)
- road graph checks: isolated nodes, connected components, arcs with a reverse twin, self-loops, duplicate arcs
- reference route checks: is it a connected chain of arcs, and how far are the GPS points from it

In [ ]:
def analyse_trip(part, tdir):
    trip = load_trip(tdir)
    tr, nd, ar = trip.track, trip.nodes, trip.arcs
    n_pts, n_nodes, n_arcs = len(tr), len(nd), len(ar)
    row = dict(partition=part, trip_id=trip.trip_id, n_points=n_pts, n_nodes=n_nodes, n_arcs=n_arcs,
               has_route=trip.route is not None,
               n_route_arcs=0 if trip.route is None else len(trip.route))

    # file availability and sizes (KB)
    for ext in ("track", "nodes", "arcs", "route"):
        f = Path(tdir) / f"{trip.trip_id}.{ext}"
        row[f"kb_{ext}"] = round(f.stat().st_size / 1024, 1) if f.exists() else np.nan

    # ---- GPS validity and timing
    row["nan_cells"] = int(tr[["lon", "lat", "t"]].isna().sum().sum())
    row["invalid_coords"] = int(((tr.lon.abs() > 180) | (tr.lat.abs() > 90) |
                                 ((tr.lon == 0) & (tr.lat == 0))).sum())
    t = tr.t.to_numpy(); dt = np.diff(t)
    row["non_increasing_t"] = int((dt <= 0).sum())
    row["duration_h"] = round((t[-1] - t[0]) / 3600, 3)
    row["median_dt_s"] = float(np.median(dt))
    row["irregular_pct"] = round(100 * float((np.abs(dt - 1.0) > 1e-6).mean()), 2)
    row["max_gap_s"] = float(dt.max())
    row["n_gaps_gt_60s"] = int((dt > 60).sum())
    row["dt_eq1"] = int((np.abs(dt - 1) <= 1e-6).sum())
    row["dt_1_2"] = int(((dt > 1 + 1e-6) & (dt <= 2)).sum())
    row["dt_2_10"] = int(((dt > 2) & (dt <= 10)).sum())
    row["dt_10_60"] = int(((dt > 10) & (dt <= 60)).sum())
    row["dt_gt60"] = int((dt > 60).sum())

    # ---- extent
    row["lon_mean"], row["lat_mean"] = float(tr.lon.mean()), float(tr.lat.mean())
    row["bbox_diag_km"] = round(float(np.hypot(np.ptp(tr.x), np.ptp(tr.y))) / 1000, 3)
    row["path_km"] = round(float(np.hypot(np.diff(tr.x), np.diff(tr.y)).sum()) / 1000, 3)

    # ---- road graph
    u, v = ar.from_node.to_numpy(), ar.to_node.to_numpy()
    used = np.unique(np.concatenate([u, v]))
    row["isolated_nodes"] = int(n_nodes - len(used))
    row["isolated_pct"] = round(100 * (n_nodes - len(used)) / n_nodes, 2)
    g = coo_matrix((np.ones(n_arcs), (u, v)), shape=(n_nodes, n_nodes)).tocsr()
    _, lab = connected_components(g, directed=False)
    sizes = np.bincount(lab[used])
    sizes = sizes[sizes > 0]
    row["n_components"] = int(len(sizes))
    row["largest_comp_pct"] = round(100 * sizes.max() / len(used), 2)
    key = u.astype(np.int64) * n_nodes + v
    rkey = v.astype(np.int64) * n_nodes + u
    row["reverse_twin_pct"] = round(100 * float(np.isin(key, rkey).mean()), 2)
    row["self_loops"] = int((u == v).sum())
    row["duplicate_arcs"] = int(n_arcs - len(np.unique(key)))

    # ---- reference route
    dist = np.array([], dtype=np.float32)
    row.update(route_contig_pct=np.nan, dist_med_m=np.nan, dist_p95_m=np.nan, dist_max_m=np.nan,
               pct_gt_50m=np.nan, pct_gt_100m=np.nan)
    if trip.route is not None and len(trip.route) > 0:
        r = trip.route
        row["route_contig_pct"] = round(100 * float((v[r[:-1]] == u[r[1:]]).mean()), 2) if len(r) > 1 else 100.0
        ra = np.unique(r)
        segs = np.stack([np.column_stack([ar.x1.to_numpy()[ra], ar.y1.to_numpy()[ra]]),
                         np.column_stack([ar.x2.to_numpy()[ra], ar.y2.to_numpy()[ra]])], axis=1)
        tree = shapely.STRtree(shapely.linestrings(segs))
        _, d = tree.query_nearest(shapely.points(tr.x.to_numpy(), tr.y.to_numpy()),
                                  return_distance=True, all_matches=False)
        dist = d.astype(np.float32)
        row.update(dist_med_m=round(float(np.median(dist)), 2), dist_p95_m=round(float(np.percentile(dist, 95)), 2),
                   dist_max_m=round(float(dist.max()), 2),
                   pct_gt_50m=round(100 * float((dist > 50).mean()), 2),
                   pct_gt_100m=round(100 * float((dist > 100).mean()), 2))
    return row, dist

rows, dists = [], {}
t0 = time.time()
for part, pdir in PARTS.items():
    for d in list_trip_dirs(pdir):
        row, dist = analyse_trip(part, d)
        rows.append(row); dists[row["trip_id"]] = dist
inv = pd.DataFrame(rows)
inv.to_csv(RESULTS / "dataset_inventory.csv", index=False)
print(f"{len(inv)} trips analysed in {time.time()-t0:.0f}s -> results/dataset_inventory.csv")
inv.head(3)

## 3. Dataset inventory

In [ ]:
summ = inv.groupby("partition").agg(
    trips=("trip_id", "count"), points=("n_points", "sum"),
    median_points=("n_points", "median"), min_points=("n_points", "min"), max_points=("n_points", "max"),
    median_nodes=("n_nodes", "median"), median_arcs=("n_arcs", "median"), max_arcs=("n_arcs", "max"),
    trips_with_route=("has_route", "sum"), total_MB=("kb_arcs", lambda s: 0))
size_cols = ["kb_track", "kb_nodes", "kb_arcs", "kb_route"]
summ["total_MB"] = inv.groupby("partition")[size_cols].sum().sum(axis=1).div(1024).round(1)
print("Trips with all 4 files:", int((inv[size_cols].notna().all(axis=1)).sum()), "of", len(inv))
summ.T

## 4. Missing / invalid values, timestamps and gaps
Nominal sampling is 1 Hz, so any time step that is not exactly 1 s counts as *irregular*.

In [ ]:
print("NaN cells in tracks:        ", int(inv.nan_cells.sum()))
print("Invalid coordinates:        ", int(inv.invalid_coords.sum()))
print("Non-increasing timestamps:  ", int(inv.non_increasing_t.sum()), "(trips:", inv[inv.non_increasing_t > 0].trip_id.tolist(), ")")
tot = inv[["dt_eq1", "dt_1_2", "dt_2_10", "dt_10_60", "dt_gt60"]].sum()
print("\nPooled time steps (train + student_test):")
print((100 * tot / tot.sum()).round(2).rename("percent").to_frame().T.to_string())
print("\nTrips with the most irregular time steps:")
print(inv.nlargest(8, "irregular_pct")[["partition", "trip_id", "n_points", "irregular_pct", "max_gap_s", "n_gaps_gt_60s"]].to_string(index=False))
print("\nTrips with the largest single time gap (hours):")
print(inv.assign(max_gap_h=(inv.max_gap_s / 3600).round(2)).nlargest(5, "max_gap_s")[["partition", "trip_id", "max_gap_h"]].to_string(index=False))

## 5. Road graph and reference route

In [ ]:
cols = ["n_nodes", "n_arcs", "isolated_pct", "n_components", "largest_comp_pct", "reverse_twin_pct",
        "self_loops", "duplicate_arcs", "route_contig_pct"]
print(inv[cols].describe().loc[["min", "50%", "mean", "max"]].round(2).T.to_string())
print("\nTrips where the reference route is NOT a fully connected chain of arcs:")
print(inv[inv.route_contig_pct < 100][["partition", "trip_id", "n_route_arcs", "route_contig_pct"]].to_string(index=False) or "none")

## 6. Distance from GPS points to their own reference route

In [ ]:
alld = np.concatenate([dists[k] for k in dists if len(dists[k])])
print(f"Pooled over {len(alld):,} points -> median {np.median(alld):.2f} m, 95th pct {np.percentile(alld,95):.1f} m, max {alld.max():.0f} m")
print(f"Median of the 90 per-trip medians: {inv.dist_med_m.median():.2f} m  (pooled value is higher because a few off-route trips add many far points)")
print("\nTrips with the largest median distance to route:")
print(inv.nlargest(6, "dist_med_m")[["partition", "trip_id", "dist_med_m", "dist_p95_m", "pct_gt_100m"]].to_string(index=False))

## 7. Anomaly log -> `results/data_quality.csv`

In [ ]:
RULES = [  # (issue, column, test, threshold, explanation)
    ("off_route",            "dist_med_m",       lambda s, t: s > t, 50,    "median GPS-to-reference-route distance above threshold (m)"),
    ("large_time_gap",       "max_gap_s",        lambda s, t: s > t, 3600,  "single time gap above threshold (s)"),
    ("irregular_timesteps",  "irregular_pct",    lambda s, t: s > t, 30,    "share of time steps that are not exactly 1 s (%)"),
    ("non_increasing_time",  "non_increasing_t", lambda s, t: s > t, 0,     "timestamps not strictly increasing (count)"),
    ("missing_or_invalid",   "nan_cells",        lambda s, t: s > t, 0,     "NaN cells in the track (count)"),
    ("invalid_coordinates",  "invalid_coords",   lambda s, t: s > t, 0,     "points outside valid lon/lat range (count)"),
    ("route_not_connected",  "route_contig_pct", lambda s, t: s < t, 100,   "consecutive route arcs that do not share a node: % connected below 100"),
    ("no_route_file",        "n_route_arcs",     lambda s, t: s <= t, 0,    "no .route file or empty (count of route arcs)"),
]
log = []
for issue, col, test, thr, note in RULES:
    hit = inv[test(inv[col], thr)]
    for _, r in hit.iterrows():
        log.append(dict(partition=r.partition, trip_id=r.trip_id, issue=issue, value=r[col], threshold=thr, note=note))
dq = pd.DataFrame(log).sort_values(["issue", "trip_id"]).reset_index(drop=True)
dq.to_csv(RESULTS / "data_quality.csv", index=False)
print(dq.groupby("issue").trip_id.apply(lambda s: ", ".join(s)).to_string())
print("\nSaved", len(dq), "rows -> results/data_quality.csv")

## 8. Figures for report section 2

In [ ]:
# Fig 1: size of the trips
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, c, lab in zip(ax, ["n_points", "n_arcs", "path_km"], ["GPS points per trip", "Road arcs per trip", "Travelled distance per trip (km)"]):
    bins = np.logspace(np.log10(inv[c].min()), np.log10(inv[c].max()), 20)
    for part, col in (("train", "tab:blue"), ("student_test", "tab:orange")):
        a.hist(inv[inv.partition == part][c], bins=bins, alpha=0.7, label=part, color=col)
    a.set_xscale("log"); a.set_xlabel(lab); a.set_ylabel("trips")
ax[0].legend(); fig.tight_layout(); fig.savefig(FIGS / "eda_trip_sizes.png"); plt.show()

# Fig 2: where the trips are (why one fixed projection cannot work)
fig, a = plt.subplots(figsize=(9, 4.5))
for part, col in (("train", "tab:blue"), ("student_test", "tab:orange")):
    s = inv[inv.partition == part]; a.scatter(s.lon_mean, s.lat_mean, s=18, c=col, label=part, alpha=0.8)
a.set_xlabel("longitude"); a.set_ylabel("latitude"); a.set_title("Trip locations (centroid of each trip)"); a.legend()
fig.tight_layout(); fig.savefig(FIGS / "eda_trip_locations.png"); plt.show()

# Fig 3: sampling regularity
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
lab = ["= 1 s", "1-2 s", "2-10 s", "10-60 s", "> 60 s"]
ax[0].bar(lab, tot.values, color="tab:blue"); ax[0].set_yscale("log"); ax[0].set_ylabel("time steps (log)"); ax[0].set_title("Pooled time steps between GPS points")
top = inv.nlargest(12, "irregular_pct").iloc[::-1]
ax[1].barh(top.trip_id, top.irregular_pct, color="tab:red"); ax[1].set_xlabel("% of steps that are not 1 s"); ax[1].set_title("Most irregular trips")
fig.tight_layout(); fig.savefig(FIGS / "eda_sampling.png"); plt.show()

# Fig 4: distance to reference route
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(np.clip(alld, 0, 60), bins=60, color="tab:green"); ax[0].set_yscale("log")
ax[0].set_xlabel("distance to own route (m, clipped at 60)"); ax[0].set_ylabel("GPS points (log)")
ax[0].axvline(np.median(alld), color="k", ls="--", label=f"median {np.median(alld):.1f} m"); ax[0].legend()
top = inv.nlargest(12, "dist_med_m").iloc[::-1]
ax[1].barh(top.trip_id, top.dist_med_m, color="tab:red"); ax[1].set_xlabel("median distance to route (m)"); ax[1].set_title("Trips furthest from their reference route")
fig.tight_layout(); fig.savefig(FIGS / "eda_dist_to_route.png"); plt.show()
print("figures saved in", FIGS)

## 9. Folium maps
Layers: raw GPS (blue), road network (grey), reference route (red). Use the layer control (top right) to switch them.
Map 1 is a typical trip; map 2 is the trip furthest from its reference route.

In [ ]:
def eda_map(trip_dir, fname, max_gps=1500, max_arcs=20000, pad=0.004):
    trip = load_trip(trip_dir); tr, nd, ar = trip.track, trip.nodes, trip.arcs
    u, v = ar.from_node.to_numpy(), ar.to_node.to_numpy()
    nlat, nlon = nd.lat.to_numpy(), nd.lon.to_numpy()
    inb = ((nd.lon.between(tr.lon.min() - pad, tr.lon.max() + pad)) &
           (nd.lat.between(tr.lat.min() - pad, tr.lat.max() + pad))).to_numpy()
    sel = np.where(inb[u] & inb[v])[0]
    sampled = len(sel) > max_arcs
    if sampled:
        sel = np.sort(rng.choice(sel, max_arcs, replace=False))
    seg = lambda i: [(float(nlat[u[i]]), float(nlon[u[i]])), (float(nlat[v[i]]), float(nlon[v[i]]))]

    m = folium.Map(location=[float(tr.lat.mean()), float(tr.lon.mean())], zoom_start=14, tiles="OpenStreetMap")
    g_net = folium.FeatureGroup(name=f"Road network (.arcs){' - random sample' if sampled else ''}", show=True)
    folium.PolyLine([seg(i) for i in sel], color="#777777", weight=1, opacity=0.6).add_to(g_net)
    g_route = folium.FeatureGroup(name="Reference route (.route)", show=True)
    if trip.route is not None:
        folium.PolyLine([seg(i) for i in trip.route], color="red", weight=4, opacity=0.8).add_to(g_route)
    g_gps = folium.FeatureGroup(name="Raw GPS points (.track)", show=True)
    stride = max(1, len(tr) // max_gps)
    for i in range(0, len(tr), stride):
        folium.CircleMarker((float(tr.lat[i]), float(tr.lon[i])), radius=2, color="blue", fill=True,
                            tooltip=f"point {i}, t={tr.t[i]:.0f}s").add_to(g_gps)
    for g in (g_net, g_route, g_gps):
        g.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[float(tr.lat.min()), float(tr.lon.min())], [float(tr.lat.max()), float(tr.lon.max())]])
    m.save(MAPS / fname)
    print(f"saved reports/maps/{fname}  (trip {trip.trip_id}: {len(tr)} GPS pts, {len(sel)} arcs drawn)")
    return m

tr_ok = inv[(inv.partition == "train") & inv.n_points.between(600, 1500) & (inv.n_arcs < 60000)]
normal_id = tr_ok.nsmallest(1, "dist_med_m").trip_id.iloc[0]
odd_id = inv[inv.partition == "train"].nlargest(1, "dist_med_m").trip_id.iloc[0]
print("typical trip:", normal_id, "| most off-route trip:", odd_id)
map_normal = eda_map(PARTS["train"] / normal_id, f"eda_trip_{normal_id}.html")
map_normal

In [ ]:
map_odd = eda_map(PARTS["train"] / odd_id, f"eda_trip_{odd_id}_offroute.html")
map_odd

## 10. Numbers for the report
Everything below is read from the CSVs written above.

In [ ]:
n_tr = (inv.partition == "train").sum(); n_st = (inv.partition == "student_test").sum()
print(f"Trips: {n_tr} train, {n_st} student_test | GPS points: {inv.n_points.sum():,} | arcs per trip: median {int(inv.n_arcs.median()):,}, max {int(inv.n_arcs.max()):,}")
print(f"Time steps exactly 1 s: {100*tot['dt_eq1']/tot.sum():.1f}% | trips with a gap > 60 s: {(inv.n_gaps_gt_60s>0).sum()} | largest gap: {inv.max_gap_s.max()/3600:.1f} h")
print(f"Arcs with a reverse twin: median {inv.reverse_twin_pct.median():.1f}% (min {inv.reverse_twin_pct.min():.1f}%)")
print(f"Isolated nodes: median {inv.isolated_pct.median():.2f}% | trips with more than one component: {(inv.n_components>1).sum()}")
print(f"GPS to own route: median of per-trip medians {inv.dist_med_m.median():.2f} m | pooled median {np.median(alld):.2f} m, p95 {np.percentile(alld,95):.1f} m")
print(f"Reference routes that are not fully connected chains: {(inv.route_contig_pct<100).sum()} trips")
print("Flagged trips:", dq.trip_id.nunique(), "| issues:", dq.issue.value_counts().to_dict())